# copyKAT Predictions

copyKAT classifies each cell as `aneuploid`, `diploid` or `not.defined` from its inferred
copy-number profile. It is run per patient by `S10a_run_copykat.py`. This notebook (step 2) 
imports those predictions into the main object and compares them with the inferCNV-based call from S09 (`cell_type_new`).
Agreement between two independent CNV methods supports the malignant annotation;
disagreement flags samples or cell types to inspect.

**Input.** Main h5ad object, and the step-1 output folder (one `*_copykat_prediction.txt`
per patient plus `copykat_run_params.json`).
**Output.** The object with `copykat_pred`; coverage and concordance tables; UMAPs and
per-sample bar plots.

Reference copyKAT: https://github.com/navinlabcode/copykat

In [ ]:
# !pip install --quiet scanpy
# %cd <project>/notebooks

In [ ]:
import os
import re
from glob import glob

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add.h5ad"
output_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT.h5ad"

figures_output_dir = "./../figures-cellbender/integration/3_annotation/2_clean/Figure_1"
tables_output_dir = "./../results/copyKAT"

# step-1 output (S10a_run_copykat.py): <copykat_dir>/<patient>/copykat_result_<patient>_copykat_prediction.txt
copykat_dir = "./copyKAT"
prediction_glob = "*/copykat_result_*_copykat_prediction.txt"
# earlier manual runs were organised as batch*/<patient>_norm/: use "batch*/*_norm/copykat_result_*_copykat_prediction.txt"
patient_from_filename = r"copykat_result_(.+)_copykat_prediction\.txt$"

pred_col = "copykat_pred"
patient_col = "ID"                      # copyKAT was run per patient (tumour + adjacent normal together)
sample_col = "ID_2"
label_col = "cell_type"
inferCNV_col = "cell_type_new"          #  call (Malignant / Non-malignant epithelial / other labels)
copykat_colors = {"aneuploid": "#d62728", "diploid": "#1f77b4", "not.defined": "#bdbdbd", "not_run": "#f0f0f0"}

patients_to_plot = [f"TN{i:03d}" for i in range(1, 10)]   # per-patient UMAPs (paired tumour / adjacent normal)

# display order of samples in bar plots; samples not listed are appended at the end
sample_order = [
    "TN008_AN", "TN004_AN", "TN006_AN", "TN007_AN", "TN001_AN", "TN002_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN001", "TN002", "TN005", "TN003", "TN009",
    "D003", "D002", "D001",
    "P003", "P002", "P006", "P008", "P012", "P005", "P013", "P001", "P009", "P011",
    "O001", "E001",
    "R006", "R003", "R001", "R010", "R009", "R007", "R005", "R008", "R012", "R004", "R002", "R011",
]

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 1. Load the main object and the copyKAT run parameters

In [ ]:
adata = sc.read(input_path)

params_file = os.path.join(copykat_dir, "copykat_run_params.json")
if os.path.exists(params_file):
    print(open(params_file).read())
else:
    print(f"{params_file} not found (results from a run without the step-1 script)")
failed_file = os.path.join(copykat_dir, "failed_patients.txt")
if os.path.exists(failed_file):
    print("patients that failed in step 1:")
    print(open(failed_file).read())

## 2. Read copyKAT predictions

In [ ]:
files = sorted(glob(os.path.join(copykat_dir, prediction_glob)))
assert files, f"no prediction files found under {copykat_dir}/{prediction_glob}"

preds = []
for f in files:
    df = pd.read_csv(f, sep="\t", index_col=0)
    df.columns = [pred_col]
    df["file_patient"] = re.search(patient_from_filename, os.path.basename(f)).group(1)
    preds.append(df)
preds = pd.concat(preds)
print(f"{len(files)} files, {len(preds)} predictions")

dup = preds.index[preds.index.duplicated()]
assert dup.empty, f"{len(dup)} barcodes predicted in more than one file, e.g. {list(dup[:3])}"
unknown = preds.index.difference(adata.obs_names)
assert unknown.empty, f"{len(unknown)} predicted barcodes not in the atlas, e.g. {list(unknown[:3])}"
wrong = (adata.obs.loc[preds.index, patient_col].astype(str) != preds["file_patient"]).sum()
assert wrong == 0, f"{wrong} cells are in a prediction file of another patient"

preds[pred_col].value_counts()

## 3. Add predictions to the atlas

Cells without a prediction are labelled `not_run` (patient not run, or cell filtered by
copyKAT's gene-count filter). The coverage table shows which.

In [ ]:
adata.obs[pred_col] = preds[pred_col].reindex(adata.obs_names).fillna("not_run")
adata.obs[pred_col] = pd.Categorical(adata.obs[pred_col], categories=[c for c in copykat_colors if c in set(adata.obs[pred_col])])
adata.uns[f"{pred_col}_colors"] = [copykat_colors[c] for c in adata.obs[pred_col].cat.categories]

coverage = pd.crosstab(adata.obs[sample_col], adata.obs[pred_col], margins=True)
coverage["pct_predicted"] = (100 * (1 - coverage.get("not_run", 0) / coverage["All"])).round(1)
coverage.to_csv(f"{tables_output_dir}/copykat_coverage_per_{sample_col}.csv")
coverage.sort_values("pct_predicted")

## 4. Concordance with the inferCNV call

Expected if both methods agree: inferCNV-Malignant cells mostly `aneuploid`; immune,
stromal and adjacent-normal cells mostly `diploid`. Computed on cells with a copyKAT call.

In [ ]:
called = adata.obs[adata.obs[pred_col].isin(["aneuploid", "diploid"])]

by_label = pd.crosstab(called[inferCNV_col], called[pred_col], normalize="index").mul(100).round(1)
by_label["n_cells"] = called[inferCNV_col].value_counts()
by_label.sort_values("aneuploid", ascending=False).to_csv(f"{tables_output_dir}/copykat_vs_{inferCNV_col}.csv")
display(by_label.sort_values("aneuploid", ascending=False))

is_malig = called[inferCNV_col] == "Malignant"
is_aneu = called[pred_col] == "aneuploid"
per_sample = pd.DataFrame({
    "n_called": called[sample_col].value_counts(),
    "pct_malignant_inferCNV": is_malig.groupby(called[sample_col], observed=True).mean().mul(100),
    "pct_aneuploid_copyKAT": is_aneu.groupby(called[sample_col], observed=True).mean().mul(100),
    "pct_agree": (is_malig == is_aneu).groupby(called[sample_col], observed=True).mean().mul(100),
}).round(1)
per_sample = per_sample.reindex([s for s in sample_order if s in per_sample.index] +
                                [s for s in per_sample.index if s not in sample_order])
per_sample.to_csv(f"{tables_output_dir}/copykat_inferCNV_concordance_per_{sample_col}.csv")
per_sample

## 5. UMAPs

In [ ]:
sc.pl.umap(adata, color=[label_col, pred_col, "EGFR_mutation", inferCNV_col], legend_fontsize="x-small",
           legend_fontoutline=True, size=20, wspace=0.4, save="_copykat_overview.pdf")

for pid in patients_to_plot:
    sub = adata[adata.obs[patient_col] == pid]
    if sub.n_obs == 0:
        print(f"{pid}: no cells, skipped")
        continue
    sc.pl.umap(sub, color=[pred_col, sample_col, label_col], legend_fontsize="x-small", wspace=0.3,
               legend_fontoutline=True, size=20, save=f"_{pid}_copykat.pdf") # supplementary figure

## 6. copyKAT call per sample

In [ ]:
props = pd.crosstab(adata.obs[sample_col], adata.obs[pred_col], normalize="index").mul(100)
order = [s for s in sample_order if s in props.index] + [s for s in props.index if s not in sample_order]
props = props.loc[order]

ax = props.plot.bar(stacked=True, edgecolor="none", width=0.85, figsize=(12, 3.5), grid=False,
                    color=[copykat_colors[c] for c in props.columns])
ax.set_ylim(0, 100)
ax.set_ylabel(f"{pred_col} (%)")
ax.set_xlabel(sample_col)
ax.legend(bbox_to_anchor=(1, 1), frameon=False, fontsize=8)
plt.savefig(f"{figures_output_dir}/barplot_{sample_col}_proportions_{pred_col}.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 7. Save

In [ ]:
adata.write(output_path)
adata.obs[pred_col].value_counts()